# Data analysis

Dataset overview for TAPP inguinal hernia repair: videos, frames, phase annotations and phase groups.

## Setup

Imports, paths and config.

## Load data

Annotation segments per surgery, annotated-frame CSVs and phase group definitions.

## Dataset summary

Surgeries, durations, frame counts per FPS and unlabelled frames.

## Video lengths

Total duration per surgery and its spread across the dataset.

## Surgery length differences

How much the video length varies between surgeries: each surgery against the median and the shortest one.

In [ ]:
import subprocess

import matplotlib.pyplot as plt
import pandas as pd

from surgphase.config import PATIENTS, VIDEOS_DIR


def video_minutes(path):
    command = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path),
    ]
    output = subprocess.run(command, capture_output=True, text=True, check=True).stdout
    return float(output) / 60


lengths = pd.Series(
    {
        surgery: video_minutes(VIDEOS_DIR / f"{surgery}.mp4")
        for surgery in PATIENTS
        if (VIDEOS_DIR / f"{surgery}.mp4").is_file()
    },
    name="minutes",
).sort_values()
missing = sorted(set(PATIENTS) - set(lengths.index))

median = lengths.median()
length_table = pd.DataFrame(
    {
        "minutes": lengths,
        "vs median (min)": lengths - median,
        "vs median (%)": (lengths / median - 1) * 100,
        "x shortest": lengths / lengths.min(),
    }
).round(1)

fig, ax = plt.subplots(figsize=(7, 0.45 * len(lengths) + 1))
ax.barh(lengths.index, lengths, height=0.6, color="#2a78d6")
ax.axvline(median, color="#52514e", linewidth=1, linestyle="--")
ax.annotate(
    f"median {median:.1f} min",
    (median, 1),
    xycoords=("data", "axes fraction"),
    xytext=(4, -2),
    textcoords="offset points",
    va="top",
    fontsize=9,
    color="#52514e",
)
for surgery in (lengths.index[0], lengths.index[-1]):
    ax.annotate(
        f"{lengths[surgery]:.1f}",
        (lengths[surgery], surgery),
        xytext=(4, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color="#0b0b0b",
    )
ax.set_xlabel("Video length (min)", color="#898781")
ax.set_title("Surgery length per video", loc="left", color="#0b0b0b")
ax.tick_params(colors="#898781", length=0)
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.show()

print(
    f"range {lengths.min():.1f}-{lengths.max():.1f} min, "
    f"longest is {lengths.max() / lengths.min():.1f}x the shortest, "
    f"coefficient of variation {lengths.std() / lengths.mean():.0%}"
)
if missing:
    print(f"no video for {missing}")
length_table

## Phase taxonomy

Phases per group and how `merged6&7` maps onto `default`.

## Ground-truth timelines

Phase sequence per surgery over time.

## Phase distribution

Frame share per phase, overall and per surgery. Shows class imbalance.

## Phase durations

Duration of each phase across surgeries, absolute and normalised by surgery length.

## Phase occurrence and order

Which phases appear in each surgery, how often they recur and whether the canonical order holds.

## Phase transitions

Transition counts between phases. The empirical prior behind the HMM.

## Segment lengths

Distribution of contiguous segment lengths per phase. Informs smoothing windows.

## Sample frames

Example frames per phase to show visual similarity between phases.

## Leave-one-surgery-out folds

Phase distribution of each held-out surgery versus its training set.